# ADS v2: Van acquisition / renewal / in-life events

Builds 81 event families: `VR1a`, `VR1b`, `VR1.E1`, `VR2a`, `VR2b`, `VR3b.F1`, `VR3a`, `VR3b`, `VR3.O1`, `VR4`, `VR5`, `VR3.B1.1`, `VR3.B1.1.F1`, `VR3.B1.2`, `VR3.B1.2.F1`, `VR3.B1.O1`, `VR3.B1.O3`, `VR4.B2`, `VR4.B2.F1`, `VR4.B2.4`, `VR4.B2.1`, `VR4.B2.2`, `VR4.B2.5`, `VR4.B2.3`, `VR4.B2.6`, `VR4.B2.O1`, `VR6.B1.1`, `VA1`, `VA0`, `VA0.F1`, `VA5`, `VA5.B1.4`, `VA6`, `VA6.B1.F1`, `VA6.B1.1`, `VA6.B1.4`, `VA6.B1.2`, `VA6.B1.3`, `VA6.B1.5`, `VA6.B1.6`, `VA6.B1.O1`, `VA7`, `VA7.B1.1`, `VA7.B1.2`, `VA7.B1.3`, `VA7.B1.5`, `VM1`, `VM1.F1`, `VM3`, `VM4c`, `VM2.O1`, `VM6`, `VM6.B1.1`, `VM6.B1.2`, `VM3.B1.1`, `VM3.B1.2`, `VM3.B1.3`, `VM3.B1.6`, `VM3.B1.4`, `VM3.B1.5`, `VM3.B1.F1`, `VM3.B1.O1`, `VC1a`, `VC2`, `VC3`, `VC4`, `VC5`, `VC6`, `VARR.1`, `VARR.2`, `VARR.3`, `VARR.4`, `VARR.O1`, `VARR.O2`, `VCL1a`, `VCL1b`, `VCL1c`, `VD1`, `VD1.F1`, `VD3.F1`, `VD.V1`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("17_Van", "started")


## Shared: van_live_policy

Van policies open at a month end (PolicyMonthEnd OpenNum = 1), the replacement for tmp.CurrentPolicy and the pii phone lists (R-1). An event on date d belongs to a Van policy open at the month end of d or of the month before (so a policy cancelled mid-month still counts that month).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_live_policy                                                             */
/*  Description : Van policies open at each month end around the window (client code, month end).       */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyMonthEnd                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_live_policy as
select  distinct
        upper(trim(PolicyCode))                                         as PolicyCode,
        upper(trim(ClientCode))                                         as ClientCode,
        cast(EffectiveDate as date)                                     as EffectiveDate
from    ${ads.ods}.PolicyMonthEnd
where   PolicyTypeGroup = 'Van'
and     OpenNum = 1
and     cast(EffectiveDate as date) >= add_months(date '${ads.start_time}', -2)
and     cast(EffectiveDate as date) <  add_months(date '${ads.end_time}', 2)

## Shared: van_calls

Inbound conversations on the Van MTA (vehicle change), claims and Documents_Out queues, or wrap-up coded as a claim, whose caller holds an open Van policy at the call date (D-E1.24, shared step adsv2_call_open_policy in 10_Common: the SCV key on the caller's number at the call time, then that customer's open Van policies). The lowest policy code is kept when the caller holds several Van policies (VanPolicyCount says how many).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_caller_policy                                                           */
/*  Description : Inbound conversations of interest x the caller's Van policy open at the call date (D- */
/*              : E1.24; lowest code).                                                                  */
/*              :                                                                                       */
/*              : Reads: ods_v2.CallSession, stg.adsv2_call_open_policy                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_caller_policy as
select  c.ConversationId,
        min(c.ConversationStartTime)                                    as ConversationStartTime,
        max(c.AniPhoneKey)                                              as CallerPhoneKey,
        min(o.PolicyCode)                                               as PolicyCode,
        max(o.Policies)                                                 as VanPolicyCount
from    ${ads.ods}.CallSession c
join    ${ads.stg}.adsv2_call_open_policy o
        on  o.ConversationId = c.ConversationId and o.PolicyTypeGroup = 'Van'
where   c.OriginatingDirection = 'inbound'
and     c.AniPhoneKey is not null
and     c.ConversationStartTime >= timestamp '${ads.start_time}' and c.ConversationStartTime < timestamp '${ads.end_time}'
and     (c.QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                         'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver',
                         'INBOUND_Claims', 'INBOUND_Documents_Out')
         or c.WrapupCodeName in ('Claims', 'Breakdown', 'Windscreen', 'Claims / Windscreen / Breakdown'))
group by c.ConversationId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_claim_call                                                              */
/*  Description : Van claims calls (D-E1.11 Motor rule): wrap-up coded Claims / Breakdown / Windscreen  */
/*              : on any queue but QD_*, plus INBOUND_Claims calls with no wrap-up code. ClaimType a =  */
/*              : claim (also the merged code from 2026-04 and uncoded calls), b = breakdown, c = winds */
/*              : creen.                                                                                */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_van_caller_policy, ods_v2.CallSession                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_claim_call as
select  v.ConversationId,
        v.ConversationStartTime,
        v.PolicyCode,
        v.CallerPhoneKey,
        case when max(case when c.WrapupCodeName in ('Claims', 'Claims / Windscreen / Breakdown') and coalesce(c.QueueName, '') not like 'QD%' then 1 else 0 end) = 1 then 'a'
             when max(case when c.WrapupCodeName = 'Breakdown' and coalesce(c.QueueName, '') not like 'QD%' then 1 else 0 end) = 1 then 'b'
             when max(case when c.WrapupCodeName = 'Windscreen' and coalesce(c.QueueName, '') not like 'QD%' then 1 else 0 end) = 1 then 'c'
             when max(case when c.QueueName = 'INBOUND_Claims' then 1 else 0 end) = 1
              and max(case when coalesce(c.WrapupCodeName, '') not in ('', 'ININ-WRAP-UP-TIMEOUT', 'Default Wrap-Up Code') then 1 else 0 end) = 0
             then 'a' end                                               as ClaimType
from    ${ads.stg}.adsv2_van_caller_policy v
join    ${ads.ods}.CallSession c        on c.ConversationId = v.ConversationId
group by v.ConversationId, v.ConversationStartTime, v.PolicyCode, v.CallerPhoneKey

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_docs_call                                                               */
/*  Description : Van document request calls (D-E1.5 Motor rule): inbound voice conversations on INBOUN */
/*              : D_Documents_Out, less transfers and other topics (a transfer, claims, complaint or Ho */
/*              : me wrap-up code); the wait friction flag (D-E1.25) on the Documents_Out sessions; the */
/*              :  caller's previous request call (for repeat calls).                                   */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_van_caller_policy, ods_v2.CallSession                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_docs_call as
select  x.*,
        lag(x.ConversationStartTime) over (partition by x.CallerPhoneKey order by x.ConversationStartTime, x.ConversationId) as PreviousCallTime
from    (
        select  v.ConversationId,
                v.ConversationStartTime,
                v.PolicyCode,
                v.CallerPhoneKey,
                case when sum(case when c.QueueName = 'INBOUND_Documents_Out' then c.AcdWaitDurationMs else 0 end) > 600000
                       or sum(case when c.QueueName = 'INBOUND_Documents_Out' then c.Abandoned else 0 end) > 0
                     then 1 else 0 end                                  as WaitFriction
        from    ${ads.stg}.adsv2_van_caller_policy v
        join    ${ads.ods}.CallSession c        on c.ConversationId = v.ConversationId
        group by v.ConversationId, v.ConversationStartTime, v.PolicyCode, v.CallerPhoneKey
        having  max(case when c.QueueName = 'INBOUND_Documents_Out' and c.MediaType = 'voice' then 1 else 0 end) = 1
        and     max(case when c.WrapupCodeName like '%Transfer%' or c.WrapupCodeName like '%Home%'
                           or c.WrapupCodeName in ('Claims', 'Breakdown', 'Windscreen', 'Claims / Windscreen / Breakdown', 'Complaint')
                         then 1 else 0 end) = 0
        ) x

## Shared: van_mta

Van MTAs from the Relay accounts postings (notebook 04 cell 34, stg.JJulyMTAs: PremiumTypeGroup 'MTA' and its posting filter), one MTA per policy and posting date (D-E1.20: one event per MTA, not one per policy for five years), with the policy's status at the posting month end (one snapshot; 04 joined all 139 month ends, so its 'not cancelled or lapsed' filter filtered nothing) and the MTA's own window: from the posting to the next MTA of the policy, at most 60 days. MTA documents, uploads and workflow records are attached to the MTA whose window they fall in.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_mta                                                                     */
/*  Description : One row per Van policy and MTA posting date, with the premium / fee totals, the statu */
/*              : s at the month end and the MTA window.                                                */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyTransaction, ods_v2.Policy, ods_v2.PolicyMonthEnd                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_mta as
select  m.*,
        least(coalesce(lead(m.PostingDate) over (partition by m.PolicyCode order by m.PostingDate), date_add(m.PostingDate, 60)),
              date_add(m.PostingDate, 60))                              as MtaWindowEnd
from    (
        select  t.PolicyCode,
                max(p.PolicyId)                                         as PolicyId,
                t.PostingDate,
                sum(t.InsurerInitialNetPremium + t.InsurerInitialCommission) as CCYGrossPremium,
                sum(case when t.ClientTransType = '8' and t.DType = 5 then t.ClientAmount else t.ClientFees end) as CCYFees,
                max(s.PolicyStatusDesc)                                 as StatusAtPosting
        from    ${ads.ods}.PolicyTransaction t
        join    ${ads.ods}.Policy p             on p.PolicyCode = t.PolicyCode and p.PolicyTypeGroup = 'Van'
        left join ${ads.ods}.PolicyMonthEnd s   on s.PolicyCode = t.PolicyCode
                                               and cast(s.EffectiveDate as date) = last_day(t.PostingDate)
        where   t.PremiumTypeGroup = 'MTA'
        and     (   (t.InsurerTransType in ('X', 'Y', 'Z', '1', '3', '5', 'C', 'E') and t.TransSeq = 0)
                 or (t.ClientTransType = '1' and t.NewFlag in (0, 4) and t.InsurerStatus = '' and t.TransSeq = 0 and t.InsCoCode = 'ZZZZ')
                 or (t.ClientTransType = '5' and t.ReturnSubType = '5' and t.TransSeq = 0 and t.InsCoCode = 'ZZZZ')
                 or (t.ClientTransType = '8' and t.DType in (5, 6) and t.TransSeq = 0)
                 or (t.InsCoCode = 'KCI' and t.AddonRef <> 0 and t.ShowOnReport = 'Y'))
        and     cast(t.PostingDate as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(t.PostingDate as timestamp) < timestamp '${ads.end_time}'
        group by t.PolicyCode, t.PostingDate
        ) m

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_mta_chase_six                                                           */
/*  Description : Van MTA policies with one of the six documents outstanding on a DAY 1 chase record (0 */
/*              : 4 cell 373's population).                                                             */
/*              :                                                                                       */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_van_mta                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_mta_chase_six as
select  distinct c.PolicyCode
from    ${ads.ods}.DocumentChaseMotorVan c
left semi join ${ads.stg}.adsv2_van_mta m on m.PolicyCode = c.PolicyCode
where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
and     (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O')

## Shared: van_cancellation

Van mid-term cancellations on the R-5 rule (agreed for Motor; V14 proposes it for Van): PolicyCancellation keyed on OldPolicyCode (the cancelled policy; PolicyCode is the new CXL record), latest snapshot per policy, with the reason and cancel date. Chases and documents attach to the cancellation inside its window (cancel date - 60 to + 30 days), by policy, not by client code as 04 did. 04 also required PolicyCancelFlag = 'Y' in every month-end snapshot (stg.JulyPolicyState), which lost most customer cancellations (Motor: 113 of 2,389).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_cancellation                                                            */
/*  Description : One row per cancelled Van policy (OldPolicyCode), its latest cancellation record.     */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyCancellation, ods_v2.Policy                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_cancellation as
select  PolicyCode, PolicyId, ClientCode, PolicyCancelDate, ShortDescription
from    (
        select  upper(trim(c.OldPolicyCode))                            as PolicyCode,
                p.PolicyId,
                c.ClientCode,
                cast(c.PolicyCancelDate as timestamp)                   as PolicyCancelDate,
                c.ShortDescription,
                row_number() over (partition by upper(trim(c.OldPolicyCode))
                                   order by c.EffectiveDate desc, c.ExtractDatetime desc, c.PolicyCancellationId desc) as rn
        from    ${ads.ods}.PolicyCancellation c
        left join ${ads.ods}.Policy p   on p.PolicyCode = upper(trim(c.OldPolicyCode))
        where   c.PolicyTypeGroup = 'Van'
        and     c.OldPolicyCode is not null
        and     c.EffectiveDate < date '${ads.effective_date}'
        ) x
where   rn = 1

## Shared: van_arrears

Arrears chase SMS / email sends (Xtremepush) to clients with an open Van policy that month, one row per send and policy. 04 read ods.scvcampaignhistory (a monthly summary, confirmed incorrect) joined to tmp.CurrentPolicy (every policy the client holds now). The Motor loan-status rule (D-E1.9, dlk.cda_loanse) cannot be applied: the loan snapshot is not in ods_v2.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_arrears_send                                                            */
/*  Description : Arrears chase sends (not mop-ups or tests) per open Van policy of the client, with th */
/*              : e chase number.                                                                       */
/*              :                                                                                       */
/*              : Reads: ods_v2.MessageEvent, stg.adsv2_van_live_policy                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_arrears_send as
select  m.EventDateTime,
        m.CampaignName,
        case when m.CampaignName like 'Arrears - Chase 1 - %' then 1
             when m.CampaignName like 'Arrears - Chase 2 - %' then 2
             when m.CampaignName like 'Arrears - Chase 3 - %' then 3
             when m.CampaignName like 'Arrears - Chase 4 - %' then 4
             when m.CampaignName like 'Arrears - Chase 5 - %' then 5
             when m.CampaignName like 'Arrears - Chase 6 - %' then 6 end as ChaseNumber,
        l.PolicyCode,
        l.ClientCode
from    ${ads.ods}.MessageEvent m
join    ${ads.stg}.adsv2_van_live_policy l
        on  l.ClientCode = upper(trim(coalesce(m.ClientCode, left(m.PolicyCode, 6))))
        and l.EffectiveDate in (last_day(cast(m.EventDateTime as date)), last_day(add_months(cast(m.EventDateTime as date), -1)))
where   m.CampaignName like 'Arrears - Chase %'
and     m.CampaignName not like '%Mop%'
and     upper(m.CampaignName) not like '%TEST%'
and     m.MessageType in ('EMAIL', 'SMS')
and     m.InteractionType = 'sent'
and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'

## Shared: van_renewal_codes

Each Van renewal cycle with both its policy codes (last year's and the renewal's) as rows, so steps that match a document, upload, workflow record or portal payment on either code use an equi-join (04 unioned two queries).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_renewal_codes                                                           */
/*  Description : Van renewal cycles x each of their two policy codes.                                  */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_renewal_codes as
select  distinct b.PolicyCode, b.RenewalDate, b.TYPolicyCode, b.PolicyRetNum, k.Code
from    ${ads.stg}.adsv2_renewal_base b
lateral view explode(array(b.PolicyCode, b.TYPolicyCode)) k as Code
where   b.LYPolicyTypeGroup = 'Van'
and     k.Code is not null

## VR1a: Renewal invitation sent by EMAIL


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr1a  (VR1a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 316. The first 'Renewal Offer - Emailed Document'  */
/*              : on the policy in the cycle window. Change: one event per renewal cycle (04: per polic */
/*              : y and month, last time), the agreed base (04: stg.VanRenewalsJuly, all history).      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr1a as
select  xxhash64('VR1a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR1a'                                              as EventTypeId,
        'Renewal invitation sent by EMAIL' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     d.EventDescription like 'Renewal Offer - Emailed Document %'
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR1b: Renewal invitation sent by POST


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr1b  (VR1b)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 317. As VR1a for 'Renewal Offer - Document Transmi */
/*              : tted' (posted).                                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr1b as
select  xxhash64('VR1b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR1b'                                              as EventTypeId,
        'Renewal invitation sent by POST' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     d.EventDescription like 'Renewal Offer - Document Transmitted %'
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR1.E1: Renewal campaigns: CRM


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr1_e1  (VR1.E1)                                                        */
/*  Description : Renewals / Chill Effort. Notebook 04 cell 318. Van renewal CRM sends (email / SMS) to */
/*              :  the renewing policy, first per cycle. Changes: keyed on the policy (04 wrote the Xtr */
/*              : emepush user id into SourcePolicyReference, grain 'User'); campaigns '%Van - Renewal% */
/*              : ' (04's '%Van - Renewals%' missed the main 'Van - Renewal - nDTI' series, ~200k sends */
/*              : ).                                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyRenewal                    */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.MessageEvent                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr1_e1 as
select  xxhash64('VR1.E1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR1.E1'                                            as EventTypeId,
        'Renewal campaigns: CRM' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.MessageEvent m       on m.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     m.CampaignName like '%Van - Renewal%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR2a: Van Renewal - Customer logs in to portal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr2a  (VR2a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 319. Portal logins by the client from 60 days befo */
/*              : re to 40 days after the due date, first per cycle (04 per month).                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: RenewalPortalEvent, PolicyRenewal              */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.RenewalPortalEvent                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr2a as
select  xxhash64('VR2a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR2a'                                              as EventTypeId,
        'Van Renewal - Customer logs in to portal' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(e.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.RenewalPortalEvent e on e.PortfolioCode = left(b.PolicyCode, 6)
        where   b.LYPolicyTypeGroup = 'Van'
        and     e.EventType = 'LoginSuccess'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR2b: Van Renewal - Customer phones call cente


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr2b  (VR2b)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 320. Conversations on INBOUND_Van_Renewals, at the */
/*              : ir start. Change: source system 20 Genesys (04: 2). D-E1.24: the call's policy = the  */
/*              : caller's renewal cycle open at the call date (due - 60 to due + 30, nearest due; shar */
/*              : ed step adsv2_call_renewal_policy), null when none.                                   */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession                                      */
/*              : Reads: ods_v2.CallSession, stg.adsv2_call_renewal_policy                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr2b as
select  xxhash64('VR2b', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR2b'                                              as EventTypeId,
        'Van Renewal - Customer phones call cente' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  x.EventDateTime,
                x.SourceSystemId,
                x.EventSourceReference,
                r.PolicyCode                                                    as SourcePolicyReference
        from    (
                select  min(ConversationStartTime)                                      as EventDateTime,
                        20                                                              as SourceSystemId,
                        ConversationId                                                  as EventSourceReference
                from    ${ads.ods}.CallSession
                where   QueueName = 'INBOUND_Van_Renewals'
                and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
                group by ConversationId
                ) x
        left join ${ads.stg}.adsv2_call_renewal_policy r
                on  r.ConversationId = x.EventSourceReference and r.PolicyTypeGroup = 'Van'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3b.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3b_f1  (VR3b.F1)                                                      */
/*  Description : Renewals / Friction. Notebook 04 cell 323. INBOUND_Van_Renewals conversations that wa */
/*              : ited more than 600 seconds or were abandoned (D-E1.25; 04: abandoned only). Change: s */
/*              : ource system 20. D-E1.24: the call's policy = the caller's renewal cycle open at the  */
/*              : call date (due - 60 to due + 30, nearest due; shared step adsv2_call_renewal_policy), */
/*              :  null when none.                                                                      */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession                                      */
/*              : Reads: ods_v2.CallSession, stg.adsv2_call_renewal_policy                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3b_f1 as
select  xxhash64('VR3b.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3b.F1'                                           as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  x.EventDateTime,
                x.SourceSystemId,
                x.EventSourceReference,
                r.PolicyCode                                                    as SourcePolicyReference
        from    (
                select  min(ConversationStartTime)                                      as EventDateTime,
                        20                                                              as SourceSystemId,
                        ConversationId                                                  as EventSourceReference
                from    ${ads.ods}.CallSession
                where   QueueName = 'INBOUND_Van_Renewals'
                and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
                group by ConversationId
                having  sum(AcdWaitDurationMs) > 600000 or sum(Abandoned) > 0
                ) x
        left join ${ads.stg}.adsv2_call_renewal_policy r
                on  r.ConversationId = x.EventSourceReference and r.PolicyTypeGroup = 'Van'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3a: Customer renews ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3a  (VR3a)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 321. A renewed cycle paid on the renewal portal (p */
/*              : ayment on the old or the new policy code, 60 days before to 40 days after the due dat */
/*              : e), on the new policy at the last such payment. 04 had no date window and read stg.Va */
/*              : nRenewalsJuly (all history).                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: RenewalPortalEvent, PolicyRenewal              */
/*              : Reads: stg.adsv2_van_renewal_codes, ods_v2.RenewalPortalEvent                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3a as
select  xxhash64('VR3a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3a'                                              as EventTypeId,
        'Customer renews ONLINE' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(e.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.TYPolicyCode                                                  as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_renewal_codes b
        join    ${ads.ods}.RenewalPortalEvent e on e.PolicyCode = b.Code
        where   b.PolicyRetNum = 1
        and     e.EventType = 'PaymentSuccess'
        and     e.EventDateTime > date_sub(b.RenewalDate, 60)
        and     e.EventDateTime < date_add(b.RenewalDate, 40)
        group by b.PolicyCode, b.RenewalDate, b.TYPolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3b: Customer renews on an INBOUND call


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3b  (VR3b)                                                            */
/*  Description : Renewals / Spine. Notebook 04 cell 322. Retained van renewals on channel 'a) IB', on  */
/*              : the new policy at its sale date, one per cycle.                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal                                  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3b as
select  xxhash64('VR3b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3b'                                              as EventTypeId,
        'Customer renews on an INBOUND call' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(TYReportingSaleDate as timestamp)                          as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     PolicyRetNum = 1
        and     Channel = 'a) IB'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.O1: Policy lapsed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_o1  (VR3.O1)                                                        */
/*  Description : Renewals / Outcome. Notebook 04 cell 324, corrected (D-E1.21, as HR3.O1): the cycle l */
/*              : apsed = lost (not renewed within the window), from the cycle's own result, at due + 3 */
/*              : 0 days once final. 04 took PolicyRetNum null at the renew date.                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal, PolicySale, RenewalPortalEvent  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_o1 as
select  xxhash64('VR3.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.O1'                                            as EventTypeId,
        'Policy lapsed' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                    as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     Lost = 1
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4: Payment taken


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4  (VR4)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 325. Retained van renewals (payment taken), on the */
/*              :  new policy at its sale date, one per cycle.                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal                                  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4 as
select  xxhash64('VR4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4'                                               as EventTypeId,
        'Payment taken' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(TYReportingSaleDate as timestamp)                          as EventDateTime,
                1                                                               as SourceSystemId,
                TYPolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     PolicyRetNum = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR5: Renewal processed-  policy continued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr5  (VR5)                                                              */
/*  Description : Renewals / Spine. Notebook 04 cell 328. Retained van renewals (renewal processed), on */
/*              :  the old policy at the new sale date. The same cycles and time as VR4 (as 04; 04 keye */
/*              : d it on the old code); 04's all-history base gave 36,332 rows before the window.      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal                                  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr5 as
select  xxhash64('VR5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR5'                                               as EventTypeId,
        'Renewal processed-  policy continued' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(TYReportingSaleDate as timestamp)                          as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     PolicyRetNum = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.B1.1: Renewal date passes with no renewal


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_b1_1  (VR3.B1.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 329, corrected (D-E1.21, as HR3.B1.1): no renewal */
/*              :  by the due date, from the cycle's resolved renewal, at the due date once passed. 04  */
/*              : counted renewals not yet due and late renewals as lapses.                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal, PolicySale, RenewalPortalEvent  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_b1_1 as
select  xxhash64('VR3.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.B1.1'                                          as EventTypeId,
        'Renewal date passes with no renewal' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewalDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     Lapsed = 1
        and     RenewalDate < date '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.B1.1.F1: Van Renewal - CRM bounces


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_b1_1_f1  (VR3.B1.1.F1)                                              */
/*  Description : Renewals / Friction. Notebook 04 cell 330 (labelled R3.B1.1.F1 there). Bounced van 'L */
/*              : apsed' campaign messages to the renewing policy in the cycle window, first per cycle. */
/*              :  Change: the van renewal base (04 joined the Motor renewal base, so it found almost n */
/*              : othing: design bug list).                                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyRenewal                    */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.MessageEvent                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_b1_1_f1 as
select  xxhash64('VR3.B1.1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.B1.1.F1'                                       as EventTypeId,
        'Van Renewal - CRM bounces' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.MessageEvent m       on m.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     m.CampaignName like '%Van%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'bounce'
        and     m.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.B1.2: Lapse CRM sequence runs


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_b1_2  (VR3.B1.2)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 331. Van 'Lapsed' campaign messages sent to the r */
/*              : enewing policy in the cycle window (the lapse sequence runs), first per cycle.        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyRenewal                    */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.MessageEvent                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_b1_2 as
select  xxhash64('VR3.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.B1.2'                                          as EventTypeId,
        'Lapse CRM sequence runs' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(m.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.MessageEvent m       on m.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     m.CampaignName like '%Van%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType in ('EMAIL', 'SMS')
        and     m.InteractionType = 'sent'
        and     m.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.B1.2.F1: Unreachable on the lapse sequence


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_b1_2_f1  (VR3.B1.2.F1)                                              */
/*  Description : Renewals / Friction. Notebook 04 cell 332, on the rule Vijay set for the Motor twin ( */
/*              : 2026-10-19): the cycle's van 'Lapsed' emails were never opened or clicked, or one bou */
/*              : nced; at the last send. 04 read ods.scvcampaignhistory (confirmed incorrect) and need */
/*              : ed both no open and a bounce.                                                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyRenewal                    */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.MessageEvent                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_b1_2_f1 as
select  xxhash64('VR3.B1.2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.B1.2.F1'                                       as EventTypeId,
        'Unreachable on the lapse sequence' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(case when m.InteractionType = 'sent' then m.EventDateTime end) as EventDateTime,
                21                                                              as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.MessageEvent m       on m.PolicyCode = b.PolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     m.CampaignName like '%Van%'
        and     m.CampaignName like '%Lapsed%'
        and     m.MessageType = 'EMAIL'
        and     m.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     m.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and m.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        having  max(case when m.InteractionType = 'sent' then 1 else 0 end) = 1
        and     (max(case when m.InteractionType in ('open', 'click') then 1 else 0 end) = 0
                 or max(case when m.InteractionType = 'bounce' then 1 else 0 end) = 1)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.B1.O1: Renewed inside the window


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_b1_o1  (VR3.B1.O1)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 333, corrected (D-E1.21, V8): renewed after the  */
/*              : due date but inside the window (due + 7 days online, due + 30 call centre), at the re */
/*              : newal sale date. 04 took "this year's renew date is later than last year's", which ev */
/*              : ery renewal meets (71,115 rows).                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal, PolicySale, RenewalPortalEvent  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_b1_o1 as
select  xxhash64('VR3.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.B1.O1'                                         as EventTypeId,
        'Renewed inside the window' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(RenewedDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     RenewedInWindow = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR3.B1.O3: Lost


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr3_b1_o3  (VR3.B1.O3)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 334, corrected (D-E1.21, as HR3.B1.O3): lost = n */
/*              : ot renewed within the window, at due + 30 days once final. 04 took lapse-campaign sen */
/*              : ds to policies with no renewal sale date.                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyRenewal, PolicySale, RenewalPortalEvent  */
/*              : Reads: stg.adsv2_renewal_base                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr3_b1_o3 as
select  xxhash64('VR3.B1.O3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR3.B1.O3'                                         as EventTypeId,
        'Lost' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(date_add(RenewalDate, 30) as timestamp)                    as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base
        where   LYPolicyTypeGroup = 'Van'
        and     Lost = 1
        and     OutcomeFinal = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2: Documents requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2  (VR4.B2)                                                        */
/*  Description : Renewals / Branch. Notebook 04 cell 326. A 'C&D' (cert and disc) document emailed or  */
/*              : transmitted on the old or new policy of a retained cycle, in the cycle window, first  */
/*              : per cycle (04: last per policy; no source filter on ods.EventStream).                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_van_renewal_codes, stg.adsv2_relay_document_event                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2 as
select  xxhash64('VR4.B2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2'                                            as EventTypeId,
        'Documents requested' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_renewal_codes b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.Code
        where   b.PolicyRetNum = 1
        and     d.EventDescription like '%C&D%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.F1: Chases before the customer submits-  and documents never submitted


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_f1  (VR4.B2.F1)                                                  */
/*  Description : Renewals / Friction. Notebook 04 cell 327. Renewal van policies with any document out */
/*              : standing in the chase snapshot, one per policy and sale, at the sale date. Change: th */
/*              : e second-car certificate is tested (04 compared a text literal); the window end repla */
/*              : ces 04's 2026-07-31.                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan                          */
/*              : Reads: ods_v2.DocumentChaseMotorVan                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_f1 as
select  xxhash64('VR4.B2.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.F1'                                         as EventTypeId,
        'Chases before the customer submits-  and documents never submitted' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp)                    as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan
        where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     PolicyType = 'Renewals'
        and     PolicyTypeGroup = 'Van'
        and     (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by PolicyCode, coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01')
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_4  (VR4.B2.4)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 337. The first chase record (CreateDate) with any */
/*              :  document outstanding, renewal van policies, one per policy and sale (04 grouped by m */
/*              : onth twice). Not yet moved to the DAY 7 send date (Home rule D-E1.16; V5 open for Van */
/*              : ).                                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan                          */
/*              : Reads: ods_v2.DocumentChaseMotorVan                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_4 as
select  xxhash64('VR4.B2.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.4'                                          as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(CreateDate) as timestamp)                              as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan
        where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     PolicyType = 'Renewals'
        and     PolicyTypeGroup = 'Van'
        and     (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by PolicyCode, coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01')
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_1  (VR4.B2.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 335. A document checklist (SMS / email) saved on  */
/*              : the new policy in the cycle window, first per cycle.                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_1 as
select  xxhash64('VR4.B2.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.1'                                          as EventTypeId,
        'Document request issued' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.TYPolicyCode                                                  as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription in ('New Business - Saved Document - Document Checklist SMS',
                                       'New Business - Saved Document - Document Checklist Email')
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate, b.TYPolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_2  (VR4.B2.2)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 336. A document upload on the old or new policy i */
/*              : n the cycle window, first per cycle (04: last per month).                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentUpload, PolicyRenewal                  */
/*              : Reads: stg.adsv2_van_renewal_codes, ods_v2.DocumentUpload                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_2 as
select  xxhash64('VR4.B2.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.2'                                          as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_renewal_codes b
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = b.Code
        where   cast(u.EventTimestamp as timestamp) >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     cast(u.EventTimestamp as timestamp) <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     cast(u.EventTimestamp as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_5  (VR4.B2.5)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 338. A final chase document on the new policy in  */
/*              : the cycle window, first per cycle.                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_5 as
select  xxhash64('VR4.B2.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.5'                                          as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     d.EventDescription like '%Chase%'
        and     d.EventDescription like '%Final%'
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_3  (VR4.B2.3)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 339. A document workflow record (any outcome, as  */
/*              : 04) on the old or new policy in the cycle window, first per cycle.                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentWorkflowStatus, PolicyRenewal          */
/*              : Reads: stg.adsv2_van_renewal_codes, ods_v2.DocumentWorkflowStatus                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_3 as
select  xxhash64('VR4.B2.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.3'                                          as EventTypeId,
        'Documents validated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_renewal_codes b
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = b.Code
        where   cast(w.EventTimestamp as timestamp) >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     cast(w.EventTimestamp as timestamp) <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     cast(w.EventTimestamp as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_6  (VR4.B2.6)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 341. An accepted document on the new policy in th */
/*              : e cycle window, first per cycle.                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentWorkflowStatus, PolicyRenewal          */
/*              : Reads: stg.adsv2_renewal_base, ods_v2.DocumentWorkflowStatus                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_6 as
select  xxhash64('VR4.B2.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.6'                                          as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     w.IsAccepted = true
        and     cast(w.EventTimestamp as timestamp) >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     cast(w.EventTimestamp as timestamp) <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR4.B2.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr4_b2_o1  (VR4.B2.O1)                                                  */
/*  Description : Renewals / Outcome. Notebook 04 cell 340. A 'CXL' document emailed or transmitted on  */
/*              : the new policy of a retained cycle, in the cycle window, where the new policy had a r */
/*              : enewal chase with documents outstanding; first per cycle.                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, DocumentChaseMotorVan, Po */
/*              : licyRenewal                                                                           */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event, ods_v2.DocumentChaseMo */
/*              : torVan                                                                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr4_b2_o1 as
select  xxhash64('VR4.B2.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR4.B2.O1'                                         as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        left semi join (
                select  PolicyCode
                from    ${ads.ods}.DocumentChaseMotorVan
                where   cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
                and     PolicyType = 'Renewals'
                and     PolicyTypeGroup = 'Van'
                and     (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
                ) c     on c.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     b.PolicyRetNum = 1
        and     d.EventDescription like '%CXL%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VR6.B1.1: Cert and disc dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vr6_b1_1  (VR6.B1.1)                                                    */
/*  Description : Renewals / Branch. Notebook 04 cell 342. Any document emailed or transmitted on the n */
/*              : ew policy of a retained cycle (not an offer or a chase), in the cycle window, first p */
/*              : er cycle (04: last per month).                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyRenewal             */
/*              : Reads: stg.adsv2_renewal_base, stg.adsv2_relay_document_event                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vr6_b1_1 as
select  xxhash64('VR6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VR6.B1.1'                                          as EventTypeId,
        'Cert and disc dispatched' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                b.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_renewal_base b
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = b.TYPolicyCode
        where   b.LYPolicyTypeGroup = 'Van'
        and     b.PolicyRetNum = 1
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDescription not like 'Renewal Offer%'
        and     d.EventDescription not like 'Document Chase%'
        and     d.EventDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime <  cast(date_add(b.RenewalDate, 60) as timestamp)
        and     d.EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and d.EventDateTime < timestamp '${ads.end_time}'
        group by b.PolicyCode, b.RenewalDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA1: Quote initiated ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va1  (VA1)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 343, corrected (D-E1.22, V1): each van quote (F */
/*              : Q / QQ, not a test email) at its first start, keyed on the quote reference with the e */
/*              : mail as a hash key in the customer reference. 04 wrote the raw email into SourcePolic */
/*              : yReference, one row per email for the whole window.                                   */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Van   Source: QuoteVan                                        */
/*              : Reads: ods_v2.QuoteVan                                                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va1 as
select  xxhash64('VA1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA1'                                               as EventTypeId,
        'Quote initiated ONLINE' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(QuoteStartDateTime)                                         as EventDateTime,
                7                                                               as SourceSystemId,
                QuoteReference                                                  as SourceQuoteReference,
                max(EmailKey)                                                   as SourceCustomerReference,
                case when max(EmailKey) is not null then 'EmailKey' end         as SourceCustomerReferenceType,
                max(QuoteId)                                                    as QuoteId
        from    ${ads.ods}.QuoteVan
        where   QuoteType in ('FQ', 'QQ')
        and     IsTestEmail = 0
        and     QuoteReference is not null
        and     QuoteStartDateTime >= timestamp '${ads.start_time}' and QuoteStartDateTime < timestamp '${ads.end_time}'
        group by QuoteReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA0: Straight into the CALL CENTRE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va0  (VA0)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cells 57, 345. Inbound conversations whose last inbo */
/*              : und queue session is INBOUND_SALES_VAN. Customers who quoted online first are not exc */
/*              : luded (V3 open; Home excludes them, D-E1.18). Change: source system 20.               */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession                                      */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va0 as
select  xxhash64('VA0', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA0'                                               as EventTypeId,
        'Straight into the CALL CENTRE' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.ConversationStartTime)                                    as EventDateTime,
                20                                                              as SourceSystemId,
                c.ConversationId                                                as EventSourceReference
        from    ${ads.ods}.CallSession c
        join    (
                select  ConversationId, max(SessionIndex) as SessionIndex
                from    ${ads.ods}.CallSession
                where   QueueName is not null and OriginatingDirection = 'inbound' and ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
                group by ConversationId
                ) l     on l.ConversationId = c.ConversationId and l.SessionIndex = c.SessionIndex
        where   c.QueueName = 'INBOUND_SALES_VAN'
        and     c.ConversationStartTime >= timestamp '${ads.start_time}' and c.ConversationStartTime < timestamp '${ads.end_time}'
        group by c.ConversationId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA0.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va0_f1  (VA0.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cells 56, 344. Conversations on INBOUND_SALES_VAN */
/*              :  that waited more than 600 seconds or were abandoned (D-E1.25; 04: abandoned only). C */
/*              : hange: source system 20.                                                              */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession                                      */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va0_f1 as
select  xxhash64('VA0.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA0.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.ods}.CallSession
        where   QueueName = 'INBOUND_SALES_VAN'
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        having  sum(AcdWaitDurationMs) > 600000 or sum(Abandoned) > 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA5: Payment successful


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va5  (VA5)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cells 58, 346. New-business van sales on channel 'a) */
/*              :  Back Office', at the sale date. Change: the shared active-sales table (SaleCount 1,  */
/*              : OrgID 1; 04's stg.VanSales used PolicyCloseNum 1).                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicySale                                     */
/*              : Reads: stg.adsv2_policies_sold                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va5 as
select  xxhash64('VA5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA5'                                               as EventTypeId,
        'Payment successful' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(ReportingSaleDate) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_policies_sold
        where   PolicyTypeGroup = 'Van'
        and     ReportingSaleType = 'New Business'
        and     Channel = 'a) Back Office'
        and     cast(ReportingSaleDate as timestamp) >= timestamp '${ads.start_time}' and cast(ReportingSaleDate as timestamp) < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA5.B1.4: Chase-  inconsistency resolution


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va5_b1_4  (VA5.B1.4)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 350. The first 'PC Chase Email' on a new-busin */
/*              : ess van sale.                                                                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va5_b1_4 as
select  xxhash64('VA5.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA5.B1.4'                                          as EventTypeId,
        'Chase-  inconsistency resolution' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription = 'Document Chase - Emailed Document - PC Chase Email'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6: Documents requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_chase                                                                   */
/*  Description : Van new-business chase snapshot rows with each row's own send date (DAY 1 / 7 / 14 /  */
/*              : 20) and whether any of the documents (all) or one of the six is outstanding.          */
/*              :                                                                                       */
/*              : Reads: ods_v2.DocumentChaseMotorVan                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_chase as
select  PolicyCode,
        Campaign,
        cast(case Campaign when 'DAY 1'  then try_cast(Day1SendDate as date)
                           when 'DAY 7'  then try_cast(Day7SendDate as date)
                           when 'DAY 14' then try_cast(Day14SendDate as date)
                           when 'DAY 20' then try_cast(Day20SendDate as date) end as timestamp) as SendDate,
        case when (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O') then 1 else 0 end                     as AnyOutstanding,
        case when (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O') then 1 else 0 end                     as SixOutstanding
from    ${ads.ods}.DocumentChaseMotorVan
where   PolicyTypeGroup = 'Van'
and     PolicyType = 'New Business'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_van_first_accepted                                                          */
/*  Description : The first accepted MyChill document (workflow record) after each new-business van sal */
/*              : e.                                                                                    */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_van_first_accepted as
select  s.PolicyCode,
        min(cast(w.EventTimestamp as timestamp))                        as FirstAccepted
from    ${ads.stg}.adsv2_policies_sold s
join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
where   s.ReportingSaleType = 'New Business'
and     s.PolicyTypeGroup = 'Van'
and     w.IsAccepted = true
and     cast(w.EventTimestamp as timestamp) >= cast(s.ReportingSaleDate as timestamp)
group by s.PolicyCode

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6  (VA6)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 347, corrected (D-E1.16 for Van, as Motor A6):  */
/*              : documents requested = the DAY 1 chase send with any document outstanding, at its send */
/*              :  date (04: the sale date). The second-car certificate is tested.                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan                          */
/*              : Reads: stg.adsv2_van_chase                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6 as
select  xxhash64('VA6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6'                                               as EventTypeId,
        'Documents requested' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_chase
        where   Campaign = 'DAY 1'
        and     AnyOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.F1: Documents never submitted-  policy at risk of cancellation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_f1  (VA6.B1.F1)                                                  */
/*  Description : Acquisition / Friction. Notebook 04 cell 348, corrected (as Motor A6.B1.F1): document */
/*              : s never submitted = a new-business van policy still chased on DAY 20 with a document  */
/*              : outstanding and no accepted document by then, at the DAY 20 send date. 04 filtered Po */
/*              : licyType 'Renewals' (renewal policies counted as acquisition friction) at the earlies */
/*              : t sale date.                                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan, DocumentWorkflowStatus  */
/*              : Reads: stg.adsv2_van_chase, stg.adsv2_van_first_accepted                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_f1 as
select  xxhash64('VA6.B1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.F1'                                         as EventTypeId,
        'Documents never submitted-  policy at risk of cancellation' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.SendDate)                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_chase c
        left join ${ads.stg}.adsv2_van_first_accepted a on a.PolicyCode = c.PolicyCode
        where   c.Campaign = 'DAY 20'
        and     c.AnyOutstanding = 1
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        and     (a.FirstAccepted is null or a.FirstAccepted > c.SendDate)
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_1  (VA6.B1.1)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 351, corrected (D-E1.16 for Van, as Motor A6.B */
/*              : 1.1): the DAY 1 chase send (document request issued) for a new-business van policy, a */
/*              : t its send date. 04 took every policy in the snapshot at its sale date.               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan                          */
/*              : Reads: stg.adsv2_van_chase                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_1 as
select  xxhash64('VA6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.1'                                          as EventTypeId,
        'Document request issued' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_chase
        where   Campaign = 'DAY 1'
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_4  (VA6.B1.4)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 353, corrected (D-E1.16 for Van, as Motor A6.B */
/*              : 1.4): the first reminder is the DAY 7 chase send with a document still outstanding, a */
/*              : t its send date. 04 read DAY 1 with one of the six Home documents at the sale date.   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan                          */
/*              : Reads: stg.adsv2_van_chase                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_4 as
select  xxhash64('VA6.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.4'                                          as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_van_chase
        where   Campaign = 'DAY 7'
        and     AnyOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_2  (VA6.B1.2)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 352. The first document upload on a new-busine */
/*              : ss van sale, in the window.                                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentUpload, PolicySale                     */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentUpload                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_2 as
select  xxhash64('VA6.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.2'                                          as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     cast(u.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_3  (VA6.B1.3)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 355. The first document workflow record (any o */
/*              : utcome, as 04) on a new-business van sale. Change: the build window (04 was fixed to  */
/*              : 2026-07-01 .. 2026-08-10).                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentWorkflowStatus, PolicySale             */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_3 as
select  xxhash64('VA6.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.3'                                          as EventTypeId,
        'Documents validated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_5  (VA6.B1.5)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 354. The first 'Chase Final Notice Email' on a */
/*              :  new-business van sale (04: the last).                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_5 as
select  xxhash64('VA6.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.5'                                          as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription = 'Document Chase - Emailed Document - Chase Final Notice Email'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_6  (VA6.B1.6)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 357. The first accepted document on a new-busi */
/*              : ness van sale (04: the last).                                                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentWorkflowStatus, PolicySale             */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_6 as
select  xxhash64('VA6.B1.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.6'                                          as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     w.IsAccepted = true
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA6.B1.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va6_b1_o1  (VA6.B1.O1)                                                  */
/*  Description : Acquisition / Outcome. Notebook 04 cell 356, corrected: cancelled for non-receipt = t */
/*              : he policy's cancellation (latest record on OldPolicyCode, stg.adsv2_van_cancellation, */
/*              :  not customer-requested), at its cancel date, on a new-business van policy with an in */
/*              : surer-led cancellation email (Reg canx template, Chill's spelling 'Cancelation') afte */
/*              : r the sale. Before: dated at the first such email.                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyCancellation, PolicyDocumentEvent, Polic */
/*              : ySale                                                                                 */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_van_cancellation, stg.adsv2_relay_document_ */
/*              : event                                                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va6_b1_o1 as
select  xxhash64('VA6.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA6.B1.O1'                                         as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(x.PolicyCancelDate)                                         as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_van_cancellation x on x.PolicyCode = s.PolicyCode
        left semi join ${ads.stg}.adsv2_relay_document_event d
                on  d.PolicyCode = s.PolicyCode
                and d.PolicyTypeGroup = 'Van'
                and d.EventDescription = 'Insurer Led Cancelation - Emailed Document - Reg canx email template'
                and d.EventDateTime >= cast(s.ReportingSaleDate as timestamp)
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     not (coalesce(x.ShortDescription, '') like 'Customer%' or coalesce(x.ShortDescription, '') like 'Client%')
        and     x.PolicyCancelDate >= cast(s.ReportingSaleDate as timestamp)
        and     x.PolicyCancelDate >= timestamp '${ads.start_time}' and x.PolicyCancelDate < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA7: Cert and disc sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va7  (VA7)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 349. Certificate transmitted / posted / transmi */
/*              : ssion confirmed on a new-business van sale, first (04: last).                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va7 as
select  xxhash64('VA7', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA7'                                               as EventTypeId,
        'Cert and disc sent' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription in ('New Business - Document Transmission Confirmed - Certificate',
                                       'New Business - Document Posted Confirmation - Certificate',
                                       'New Business - Document Transmitted - Certificate')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA7.B1.1: Cert and disc dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va7_b1_1  (VA7.B1.1)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 358. The cert and disc issue letter printed (' */
/*              : CD Issue Letter CV') on a new-business van sale, first.                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va7_b1_1 as
select  xxhash64('VA7.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA7.B1.1'                                          as EventTypeId,
        'Cert and disc dispatched' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription = 'New Business - Printed Document - CD Issue Letter CV'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA7.B1.2: Document pack dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va7_b1_2  (VA7.B1.2)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 359. Terms of business emailed or transmitted  */
/*              : (the document pack) on a new-business van sale, first.                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va7_b1_2 as
select  xxhash64('VA7.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA7.B1.2'                                          as EventTypeId,
        'Document pack dispatched' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription in ('New Business - Emailed Document - 05E Terms Of Business',
                                       'New Business - Document Transmitted - 05E Terms Of Business',
                                       'New Business - Document Transmitted - Terms Of Business')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA7.B1.3: Customer receipt-  implicit


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va7_b1_3  (VA7.B1.3)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 360. Terms of business transmitted or emailed  */
/*              : (customer receipt, implicit) on a new-business van sale, first.                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va7_b1_3 as
select  xxhash64('VA7.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA7.B1.3'                                          as EventTypeId,
        'Customer receipt-  implicit' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription in ('New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Emailed Document - Terms Of Business')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VA7.B1.5: Replacement requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_va7_b1_5  (VA7.B1.5)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 361. The MTA cert and disc issue letter printe */
/*              : d ('CD Issue Letter MTA') on a new-business van sale (replacement requested), first.  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicySale                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_va7_b1_5 as
select  xxhash64('VA7.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VA7.B1.5'                                          as EventTypeId,
        'Replacement requested' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.PolicyTypeGroup = 'Van'
        and     s.ReportingSaleType = 'New Business'
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription = 'New Business - Printed Document - CD Issue Letter MTA'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM1: Customer calls Chill with a change request


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm1  (VM1)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 362. Inbound conversations on the six vehicle-change qu */
/*              : eues from a caller holding an open Van policy that month, one per conversation, with  */
/*              : the policy and the caller's phone key. 04 matched Van callers through a pii phone lis */
/*              : t built in a scratch notebook (R-1) and wrote each call twice.                        */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: ods_v2.CallSession, stg.adsv2_van_caller_policy                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm1 as
select  xxhash64('VM1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM1'                                               as EventTypeId,
        'Customer calls Chill with a change request' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.ConversationStartTime)                                    as EventDateTime,
                20                                                              as SourceSystemId,
                c.ConversationId                                                as EventSourceReference,
                v.PolicyCode                                                    as SourcePolicyReference,
                v.CallerPhoneKey                                                as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.ods}.CallSession c
        join    ${ads.stg}.adsv2_van_caller_policy v on v.ConversationId = c.ConversationId
        where   c.QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                                'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver')
        and     c.OriginatingDirection = 'inbound'
        and     c.ConversationStartTime >= timestamp '${ads.start_time}' and c.ConversationStartTime < timestamp '${ads.end_time}'
        group by c.ConversationId, v.PolicyCode, v.CallerPhoneKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM1.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm1_f1  (VM1.F1)                                                        */
/*  Description : Mta / Friction. Notebook 04 cells 52, 363. The VM1 calls that waited more than 600 se */
/*              : conds or were abandoned on the vehicle-change queues (D-E1.25; 04: abandoned with any */
/*              :  wait).                                                                               */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: ods_v2.CallSession, stg.adsv2_van_caller_policy                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm1_f1 as
select  xxhash64('VM1.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM1.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.ConversationStartTime)                                    as EventDateTime,
                20                                                              as SourceSystemId,
                c.ConversationId                                                as EventSourceReference,
                v.PolicyCode                                                    as SourcePolicyReference,
                v.CallerPhoneKey                                                as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.ods}.CallSession c
        join    ${ads.stg}.adsv2_van_caller_policy v on v.ConversationId = c.ConversationId
        where   c.QueueName in ('INBOUND_VehicleChangeAddress', 'INBOUND_VehicleChangeOther', 'INBOUND_VehicleChange_Perm',
                                'INBOUND_VehicleChange_Temp', 'INBOUND_VehicleUpdateLicence', 'INBOUND_Vehicle_Add_Driver')
        and     c.ConversationStartTime >= timestamp '${ads.start_time}' and c.ConversationStartTime < timestamp '${ads.end_time}'
        group by c.ConversationId, v.PolicyCode, v.CallerPhoneKey
        having  sum(c.AcdWaitDurationMs) > 600000 or sum(c.Abandoned) > 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3: Customer accepts


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3  (VM3)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 365. Each van MTA posting on a policy not cancelled or  */
/*              : lapsed at that month end, at the posting date (D-E1.20: 04 kept one per policy for fi */
/*              : ve years).                                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyTransaction, PolicyMonthEnd              */
/*              : Reads: stg.adsv2_van_mta                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3 as
select  xxhash64('VM3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3'                                               as EventTypeId,
        'Customer accepts' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(PostingDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_van_mta
        where   coalesce(StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM4c: Return premium


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm4c  (VM4c)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 367. Van MTA postings with zero gross premium and zero  */
/*              : fees (04's 'return premium' rule, kept; the Motor review reads a return premium as a  */
/*              : negative premium: decision needed), per MTA.                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyTransaction, PolicyMonthEnd              */
/*              : Reads: stg.adsv2_van_mta                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm4c as
select  xxhash64('VM4c', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM4c'                                              as EventTypeId,
        'Return premium' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(PostingDate as timestamp)                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_van_mta
        where   coalesce(StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     CCYGrossPremium = 0
        and     CCYFees = 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM2.O1: Cancellation-  declined or not proceeded with


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm2_o1  (VM2.O1)                                                        */
/*  Description : Mta / Outcome. Notebook 04 cell 364. A van policy with an MTA in the window, now 'Can */
/*              : celled Mid Term' or 'Lapsed for Transfer', at its cancel date when that is on or afte */
/*              : r its first MTA (04: the status in any of 139 month ends, no date link).              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyTransaction, Policy                      */
/*              : Reads: stg.adsv2_van_mta, ods_v2.Policy                                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm2_o1 as
select  xxhash64('VM2.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM2.O1'                                            as EventTypeId,
        'Cancellation-  declined or not proceeded with' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(p.PolicyCancelDate) as timestamp)                      as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(p.PolicyId)                                                 as PolicyId
        from    (select PolicyCode, min(PostingDate) as FirstPostingDate from ${ads.stg}.adsv2_van_mta group by PolicyCode) m
        join    ${ads.ods}.Policy p             on p.PolicyCode = m.PolicyCode
        where   p.PolicyStatusDesc in ('Cancelled Mid Term', 'Lapsed for Transfer')
        and     cast(p.PolicyCancelDate as date) >= m.FirstPostingDate
        and     cast(p.PolicyCancelDate as timestamp) >= timestamp '${ads.start_time}' and cast(p.PolicyCancelDate as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM6: Cert and disc sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm6  (VM6)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 368. The first document emailed or transmitted on the p */
/*              : olicy inside the MTA's window (posting to the next MTA, at most 60 days), one per MTA */
/*              : . 04 took the last document of any type over five years, per policy.                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyTransaction         */
/*              : Reads: stg.adsv2_van_mta, stg.adsv2_relay_document_event                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm6 as
select  xxhash64('VM6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM6'                                               as EventTypeId,
        'Cert and disc sent' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= cast(m.PostingDate as timestamp)
        and     d.EventDateTime <  cast(m.MtaWindowEnd as timestamp)
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM6.B1.1: Cert and disc dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm6_b1_1  (VM6.B1.1)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 376. One of six new-business pack documents (04's list */
/*              : ) emailed or transmitted inside the MTA's window, first per MTA.                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyTransaction         */
/*              : Reads: stg.adsv2_van_mta, stg.adsv2_relay_document_event                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm6_b1_1 as
select  xxhash64('VM6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM6.B1.1'                                          as EventTypeId,
        'Cert and disc dispatched' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription in ('New Business - Emailed Document - Terms Of Business',
                                       'New Business - Emailed Document - Cover Letter',
                                       'New Business - Emailed Document - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Document Transmitted - Cover Letter')
        and     d.EventDateTime >= cast(m.PostingDate as timestamp)
        and     d.EventDateTime <  cast(m.MtaWindowEnd as timestamp)
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM6.B1.2: Document pack dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm6_b1_2  (VM6.B1.2)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 377. The same rule as VM6.B1.1 (as 04: the same popula */
/*              : tion, the recon expects equal volumes).                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyTransaction         */
/*              : Reads: stg.adsv2_van_mta, stg.adsv2_relay_document_event                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm6_b1_2 as
select  xxhash64('VM6.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM6.B1.2'                                          as EventTypeId,
        'Document pack dispatched' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription in ('New Business - Emailed Document - Terms Of Business',
                                       'New Business - Emailed Document - Cover Letter',
                                       'New Business - Emailed Document - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Document Transmitted - Cover Letter')
        and     d.EventDateTime >= cast(m.PostingDate as timestamp)
        and     d.EventDateTime <  cast(m.MtaWindowEnd as timestamp)
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_1  (VM3.B1.1)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 369. An MTA followed by a document upload inside its w */
/*              : indow (a document request was issued), at the posting date (as 04's time). 04's VM3.B */
/*              : 1.1 and VM3.B1.2 were identical; VM3.B1.2 now carries the upload time.                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentUpload, PolicyTransaction              */
/*              : Reads: stg.adsv2_van_mta, ods_v2.DocumentUpload                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_1 as
select  xxhash64('VM3.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.1'                                          as EventTypeId,
        'Document request issued' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(m.PostingDate as timestamp)                                as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     cast(u.EventTimestamp as timestamp) >= cast(m.PostingDate as timestamp)
        and     cast(u.EventTimestamp as timestamp) <  cast(m.MtaWindowEnd as timestamp)
        and     cast(u.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_2  (VM3.B1.2)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 370. The first document upload inside the MTA's window */
/*              :  (04 stamped it at the posting date), one per MTA.                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentUpload, PolicyTransaction              */
/*              : Reads: stg.adsv2_van_mta, ods_v2.DocumentUpload                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_2 as
select  xxhash64('VM3.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.2'                                          as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     cast(u.EventTimestamp as timestamp) >= cast(m.PostingDate as timestamp)
        and     cast(u.EventTimestamp as timestamp) <  cast(m.MtaWindowEnd as timestamp)
        and     cast(u.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_3  (VM3.B1.3)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 372. The first document workflow record (any outcome,  */
/*              : as 04) inside the MTA's window, one per MTA (04 stamped the posting date).            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentWorkflowStatus, PolicyTransaction      */
/*              : Reads: stg.adsv2_van_mta, ods_v2.DocumentWorkflowStatus                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_3 as
select  xxhash64('VM3.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.3'                                          as EventTypeId,
        'Documents validated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     cast(w.EventTimestamp as timestamp) >= cast(m.PostingDate as timestamp)
        and     cast(w.EventTimestamp as timestamp) <  cast(m.MtaWindowEnd as timestamp)
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_6  (VM3.B1.6)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 374. The first accepted document inside the MTA's wind */
/*              : ow, one per MTA (04: the last over five years per policy).                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentWorkflowStatus, PolicyTransaction      */
/*              : Reads: stg.adsv2_van_mta, ods_v2.DocumentWorkflowStatus                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_6 as
select  xxhash64('VM3.B1.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.6'                                          as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     w.IsAccepted = true
        and     cast(w.EventTimestamp as timestamp) >= cast(m.PostingDate as timestamp)
        and     cast(w.EventTimestamp as timestamp) <  cast(m.MtaWindowEnd as timestamp)
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_4  (VM3.B1.4)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 371. Van MTA policies with one of six documents outsta */
/*              : nding on a DAY 1 chase, at the chase's sale date, one per policy and sale (as 04). Th */
/*              : e chase snapshot is the new-business / renewal chase, not an MTA document request (P9 */
/*              : ; decision needed).                                                                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan, PolicyTransaction       */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_van_mta                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_4 as
select  xxhash64('VM3.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.4'                                          as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp)                  as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        left semi join ${ads.stg}.adsv2_van_mta m on m.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Van'
        and     c.Campaign = 'DAY 1'
        and     (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O')
        group by c.PolicyCode, coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_5  (VM3.B1.5)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 375. Van MTA policies on a DAY 20 chase, at the chase' */
/*              : s sale date, one per policy and sale (as 04; P9).                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan, PolicyTransaction       */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_van_mta                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_5 as
select  xxhash64('VM3.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.5'                                          as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp)                  as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        left semi join ${ads.stg}.adsv2_van_mta m on m.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= timestamp '${ads.start_time}' and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Van'
        and     c.Campaign = 'DAY 20'
        group by c.PolicyCode, coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.F1: Chases before the customer submits-  and documents never submitted


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_f1  (VM3.B1.F1)                                                  */
/*  Description : Mta / Friction. Notebook 04 cell 366. Van MTA policies with one of seven documents ou */
/*              : tstanding (the six plus the direct debit mandate), at the latest sale date. Change: t */
/*              : he Motor / Van chase snapshot (04 read the Home snapshot, which has no Van rows: desi */
/*              : gn bug list); the window end replaces 04's 2026-07-31.                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan, PolicyTransaction       */
/*              : Reads: ods_v2.DocumentChaseMotorVan, stg.adsv2_van_mta                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_f1 as
select  xxhash64('VM3.B1.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.F1'                                         as EventTypeId,
        'Chases before the customer submits-  and documents never submitted' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(max(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01')) as timestamp)             as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.ods}.DocumentChaseMotorVan c
        left semi join ${ads.stg}.adsv2_van_mta m on m.PolicyCode = c.PolicyCode
        where   cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(coalesce(try_cast(concat(right(c.SaleDate, 4), '-', substring(c.SaleDate, 4, 2), '-', left(c.SaleDate, 2)) as date), date '1900-01-01') as timestamp) < timestamp '${ads.end_time}'
        and     c.PolicyTypeGroup = 'Van'
        and     (Comp_DDM_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O')
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VM3.B1.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vm3_b1_o1  (VM3.B1.O1)                                                  */
/*  Description : Mta / Outcome. Notebook 04 cell 373. The '10 Day CXL Letter' transmitted inside the M */
/*              : TA's window, on a policy with one of six documents outstanding on a chase; first per  */
/*              : MTA (04: the last per policy, at the posting date).                                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, DocumentChaseMotorVan, Po */
/*              : licyTransaction                                                                       */
/*              : Reads: stg.adsv2_van_mta, stg.adsv2_van_mta_chase_six, stg.adsv2_relay_document_event */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vm3_b1_o1 as
select  xxhash64('VM3.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VM3.B1.O1'                                         as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_mta m
        join    ${ads.stg}.adsv2_van_mta_chase_six c on c.PolicyCode = m.PolicyCode
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   coalesce(m.StatusAtPosting, '') not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term', 'Lapsed for Transfer')
        and     d.PolicyTypeGroup = 'Van'
        and     d.EventDescription = 'Prior Year Quotes - Document Transmitted - 10 Day CXL Letter'
        and     d.EventDateTime >= cast(m.PostingDate as timestamp)
        and     d.EventDateTime <  cast(m.MtaWindowEnd as timestamp)
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VC1a: Customer asks to cancel


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vc1a  (VC1a)                                                            */
/*  Description : Cancellation / Spine. Notebook 04 cell 378. A customer-requested van cancellation (re */
/*              : ason 'Client%' / 'Customer%'), on the cancelled policy at its cancel date. Changes (R */
/*              : -5 shape, V14 open): keyed on OldPolicyCode, latest record per policy, no month-end P */
/*              : olicyCancelFlag filter (stg.JulyPolicyState, R-1).                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyCancellation                             */
/*              : Reads: stg.adsv2_van_cancellation                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vc1a as
select  xxhash64('VC1a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VC1a'                                              as EventTypeId,
        'Customer asks to cancel' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  PolicyCancelDate                                                as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_van_cancellation
        where   (ShortDescription like 'Client%' or ShortDescription like 'Customer%')
        and     PolicyCancelDate >= timestamp '${ads.start_time}' and PolicyCancelDate < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VC2: Document chase-  reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vc2  (VC2)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 379. The first DAY 1 chase record (one of six  */
/*              : documents outstanding) on the cancelled policy inside the cancellation window (cancel */
/*              :  date - 60 to + 30 days), at the record's date. 04 matched any policy of any client i */
/*              : n the cancellation table, by client code, at the sale date.                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan, PolicyCancellation      */
/*              : Reads: stg.adsv2_van_cancellation, ods_v2.DocumentChaseMotorVan                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vc2 as
select  xxhash64('VC2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VC2'                                               as EventTypeId,
        'Document chase-  reminder' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(c.CreateDate) as timestamp)                            as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_cancellation x
        join    ${ads.ods}.DocumentChaseMotorVan c on c.PolicyCode = x.PolicyCode
        where   c.Campaign = 'DAY 1'
        and     (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O')
        and     cast(c.CreateDate as timestamp) >= x.PolicyCancelDate - interval 60 days
        and     cast(c.CreateDate as timestamp) <  x.PolicyCancelDate + interval 30 days
        and     cast(c.CreateDate as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(c.CreateDate as timestamp) < timestamp '${ads.end_time}'
        group by x.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VC3: Document chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vc3  (VC3)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 380. As VC2 for the later chase campaigns (not */
/*              :  DAY 1: escalation).                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: DocumentChaseMotorVan, PolicyCancellation      */
/*              : Reads: stg.adsv2_van_cancellation, ods_v2.DocumentChaseMotorVan                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vc3 as
select  xxhash64('VC3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VC3'                                               as EventTypeId,
        'Document chase-  escalation' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(c.CreateDate) as timestamp)                            as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_cancellation x
        join    ${ads.ods}.DocumentChaseMotorVan c on c.PolicyCode = x.PolicyCode
        where   c.Campaign <> 'DAY 1'
        and     (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O')
        and     cast(c.CreateDate as timestamp) >= x.PolicyCancelDate - interval 60 days
        and     cast(c.CreateDate as timestamp) <  x.PolicyCancelDate + interval 30 days
        and     cast(c.CreateDate as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(c.CreateDate as timestamp) < timestamp '${ads.end_time}'
        group by x.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VC4: Final notice


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vc4  (VC4)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 381. The first 'Document Chase ... Final Notic */
/*              : e' emailed or transmitted on the cancelled policy inside the cancellation window (04: */
/*              :  last, by client code).                                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyCancellation        */
/*              : Reads: stg.adsv2_van_cancellation, stg.adsv2_relay_document_event                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vc4 as
select  xxhash64('VC4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VC4'                                               as EventTypeId,
        'Final notice' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_cancellation x
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = x.PolicyCode
        where   d.EventDescription like '%Document Chase%'
        and     d.EventDescription like '%Final Notice%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= x.PolicyCancelDate - interval 60 days
        and     d.EventDateTime <  x.PolicyCancelDate + interval 30 days
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by x.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VC5: Cancellation processed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vc5  (VC5)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 382. The first 'CXL' document emailed or trans */
/*              : mitted on the cancelled policy inside the cancellation window (04: last, by client co */
/*              : de).                                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, PolicyCancellation        */
/*              : Reads: stg.adsv2_van_cancellation, stg.adsv2_relay_document_event                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vc5 as
select  xxhash64('VC5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VC5'                                               as EventTypeId,
        'Cancellation processed' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                x.PolicyCode                                                    as SourcePolicyReference,
                max(x.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_cancellation x
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = x.PolicyCode
        where   d.EventDescription like '%CXL%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= x.PolicyCancelDate - interval 60 days
        and     d.EventDateTime <  x.PolicyCancelDate + interval 30 days
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by x.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VC6: Cancellation confirmation sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vc6  (VC6)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 383. The insurer-led cancellation email (Reg c */
/*              : anx template, 'Cancelation' as Chill spells it) on a van policy, first per policy. As */
/*              :  04 it is not tied to the cancellation table (it covers insurer-led cancellations, wh */
/*              : ich customer cancellations do not get).                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent                            */
/*              : Reads: stg.adsv2_relay_document_event                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vc6 as
select  xxhash64('VC6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VC6'                                               as EventTypeId,
        'Cancellation confirmation sent' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_relay_document_event
        where   PolicyTypeGroup = 'Van'
        and     EventDescription = 'Insurer Led Cancelation - Emailed Document - Reg canx email template'
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VARR.1: Missed payment detected


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_varr_1  (VARR.1)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 384. Each 'Arrears - Chase 1' send to the client of */
/*              :  an open van policy (missed payment detected), one per send (04: the last per policy) */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyMonthEnd                   */
/*              : Reads: stg.adsv2_van_arrears_send                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_varr_1 as
select  xxhash64('VARR.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VARR.1'                                            as EventTypeId,
        'Missed payment detected' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_arrears_send
        where   ChaseNumber = 1
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, ClientCode, date_trunc('MINUTE', EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VARR.2: Arrears chase


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_varr_2  (VARR.2)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 385, on the Motor rule (D-E1.9, V15 open): each 'Ar */
/*              : rears - Chase 1' to 'Chase 5' send (not mop-ups or tests). 04 took Chase 1 only, the  */
/*              : same rows as VARR.1.                                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyMonthEnd                   */
/*              : Reads: stg.adsv2_van_arrears_send                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_varr_2 as
select  xxhash64('VARR.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VARR.2'                                            as EventTypeId,
        'Arrears chase' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_arrears_send
        where   ChaseNumber between 1 and 5
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, ClientCode, date_trunc('MINUTE', EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VARR.3: Loan defaults-  arrears not cleared


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_varr_3  (VARR.3)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 387. Each 'Arrears - Chase 5 - 28 Days Past' send ( */
/*              : loan not cleared; a chase, not the default itself: P9).                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyMonthEnd                   */
/*              : Reads: stg.adsv2_van_arrears_send                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_varr_3 as
select  xxhash64('VARR.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VARR.3'                                            as EventTypeId,
        'Loan defaults-  arrears not cleared' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_arrears_send
        where   ChaseNumber = 5
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, ClientCode, date_trunc('MINUTE', EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VARR.4: Default chase and final notice


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_varr_4  (VARR.4)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 388. Each 'Arrears - Chase 6 - 38 Days Past' send ( */
/*              : default chase and final notice; the campaign ran in Nov 2022 only).                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyMonthEnd                   */
/*              : Reads: stg.adsv2_van_arrears_send                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_varr_4 as
select  xxhash64('VARR.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VARR.4'                                            as EventTypeId,
        'Default chase and final notice' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(EventDateTime)                                              as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_arrears_send
        where   ChaseNumber = 6
        and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, ClientCode, date_trunc('MINUTE', EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VARR.O1: Arrears cleared


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_varr_o1  (VARR.O1)                                                      */
/*  Description : Arrears / Outcome. Notebook 04 cell 386. A Chase 1 send with no Chase 5 send for the  */
/*              : policy in the 35 days after it (arrears cleared), once 28 days have passed before the */
/*              :  window end, at the Chase 1 send. Change: per arrears cycle (04: any Chase 5 anywhere */
/*              :  in the window) and the window end replaces current_timestamp() (04's result depended */
/*              :  on the run date).                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: MessageEvent, PolicyMonthEnd                   */
/*              : Reads: stg.adsv2_van_arrears_send                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_varr_o1 as
select  xxhash64('VARR.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VARR.O1'                                           as EventTypeId,
        'Arrears cleared' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(a.EventDateTime)                                            as EventDateTime,
                21                                                              as SourceSystemId,
                a.PolicyCode                                                    as SourcePolicyReference,
                a.ClientCode                                                    as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_arrears_send a
        left anti join (
                select  PolicyCode, EventDateTime
                from    ${ads.stg}.adsv2_van_arrears_send
                where   ChaseNumber = 5
                ) f     on  f.PolicyCode = a.PolicyCode
                        and f.EventDateTime >  a.EventDateTime
                        and f.EventDateTime <= a.EventDateTime + interval 35 days
        where   a.ChaseNumber = 1
        and     a.EventDateTime + interval 28 days < timestamp '${ads.end_time}'
        and     a.EventDateTime >= timestamp '${ads.start_time}' and a.EventDateTime < timestamp '${ads.end_time}'
        group by a.PolicyCode, a.ClientCode, date_trunc('MINUTE', a.EventDateTime)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VARR.O2: Escalation to forced cancellation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_varr_o2  (VARR.O2)                                                      */
/*  Description : Arrears / Outcome. Notebook 04 cell 389. The first insurer-led cancellation event ('I */
/*              : nsurer Led%') on a van policy within 60 days after an arrears chase to its client. Ch */
/*              : ange: the time link (04 took any such event on a policy that ever had an arrears camp */
/*              : aign, P12).                                                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, MessageEvent, PolicyMonth */
/*              : End                                                                                   */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_van_arrears_send                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_varr_o2 as
select  xxhash64('VARR.O2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VARR.O2'                                           as EventTypeId,
        'Escalation to forced cancellation' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                d.PolicyCode                                                    as SourcePolicyReference,
                max(d.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_relay_document_event d
        join    ${ads.stg}.adsv2_van_arrears_send a on a.PolicyCode = d.PolicyCode
        where   d.PolicyTypeGroup = 'Van'
        and     d.EventDescription like 'Insurer Led%'
        and     d.EventDateTime >= a.EventDateTime
        and     d.EventDateTime <  a.EventDateTime + interval 60 days
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by d.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VCL1a: Van claim reported


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vcl1a  (VCL1a)                                                          */
/*  Description : Claims / Spine. Notebook 04 cell 390, on the Motor rule (D-E1.11): a claims call (wra */
/*              : p-up 'Claims', the merged 'Claims / Windscreen / Breakdown' from 2026-04, or an uncod */
/*              : ed INBOUND_Claims call) from a caller with an open Van policy, one per call. 04 match */
/*              : ed callers through a pii phone list (R-1), one row per policy for the whole window.   */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: stg.adsv2_van_claim_call                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vcl1a as
select  xxhash64('VCL1a', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VCL1a'                                             as EventTypeId,
        'Van claim reported' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_claim_call
        where   ClaimType = 'a'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VCL1b: Breakdown assistance requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vcl1b  (VCL1b)                                                          */
/*  Description : Claims / Spine. New (D-E1.11 Motor rule; 04 had no VCL1b). A call wrap-up coded 'Brea */
/*              : kdown' (to 2026-03) from a caller with an open Van policy.                            */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: stg.adsv2_van_claim_call                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vcl1b as
select  xxhash64('VCL1b', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VCL1b'                                             as EventTypeId,
        'Breakdown assistance requested' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_claim_call
        where   ClaimType = 'b'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VCL1c: Windscreen claim initiated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vcl1c  (VCL1c)                                                          */
/*  Description : Claims / Spine. New (D-E1.11 Motor rule; 04 had no VCL1c). A call wrap-up coded 'Wind */
/*              : screen' (to 2026-03) from a caller with an open Van policy.                           */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: stg.adsv2_van_claim_call                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vcl1c as
select  xxhash64('VCL1c', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VCL1c'                                             as EventTypeId,
        'Windscreen claim initiated' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_claim_call
        where   ClaimType = 'c'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VD1: Customer rings and asks for a document


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vd1  (VD1)                                                              */
/*  Description : Doc Requests / Spine. Notebook 04 cell 391, on the Motor rule (D-E1.5): each inbound  */
/*              : Documents_Out voice call (not a transfer or another topic) from a caller with an open */
/*              :  Van policy, at the call start. 04 kept only calls followed by a Relay document withi */
/*              : n 2 days, fixed to 2026-07-01 .. 2026-08-10, by client code.                          */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: stg.adsv2_van_docs_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vd1 as
select  xxhash64('VD1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VD1'                                               as EventTypeId,
        'Customer rings and asks for a document' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_docs_call
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VD1.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vd1_f1  (VD1.F1)                                                        */
/*  Description : Doc Requests / Friction. Notebook 04 cell 392, on D-E1.25: the VD1 calls that waited  */
/*              : more than 600 seconds on Documents_Out or were abandoned (04: abandoned, no time link */
/*              : ).                                                                                    */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: stg.adsv2_van_docs_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vd1_f1 as
select  xxhash64('VD1.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VD1.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_docs_call
        where   WaitFriction = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VD3.F1: Non-arrival-  and the customer rings again


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vd3_f1  (VD3.F1)                                                        */
/*  Description : Doc Requests / Friction. Notebook 04 cell 394, on the Motor rule (D-E1.8): a VD1 call */
/*              :  by a caller who made another VD1 call in the 14 days before (non-arrival, rings agai */
/*              : n), at the repeat call. 04: clients with calls on more than one date, any time, one p */
/*              : er client.                                                                            */
/*              :                                                                                       */
/*              : Grain: Call   Product: Van   Source: CallSession, PolicyMonthEnd, SourceCustomer      */
/*              : Reads: stg.adsv2_van_docs_call                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vd3_f1 as
select  xxhash64('VD3.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VD3.F1'                                            as EventTypeId,
        'Non-arrival-  and the customer rings again' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                CallerPhoneKey                                                  as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_van_docs_call
        where   PreviousCallTime is not null
        and     PreviousCallTime >= ConversationStartTime - interval 14 days
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## VD.V1: Replacement cert and disc


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_vd_v1  (VD.V1)                                                          */
/*  Description : Doc Requests / Spine. Notebook 04 cell 396. A cert and disc email ('Prior Year Quotes */
/*              :  - Emailed Document - C&D Email') on the caller's Van policy within 2 days after a VD */
/*              : 1 call, first per call. D-E1.7 models the Motor D.V1 on the cert and disc branch inst */
/*              : ead (to agree for Van, V15).                                                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Van   Source: PolicyDocumentEvent, CallSession, PolicyMonthE */
/*              : nd, SourceCustomer                                                                    */
/*              : Reads: stg.adsv2_van_docs_call, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_vd_v1 as
select  xxhash64('VD.V1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'VD.V1'                                             as EventTypeId,
        'Replacement cert and disc' as EventDescription,
        'Van'                                               as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                c.ConversationId                                                as EventSourceReference,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(d.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_van_docs_call c
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = c.PolicyCode
        where   d.EventDescription = 'Prior Year Quotes - Emailed Document - C&D Email'
        and     d.EventDateTime >= c.ConversationStartTime
        and     d.EventDateTime <  c.ConversationStartTime + interval 2 days
        group by c.ConversationId, c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_vr1a", "adsv2_evt_vr1b", "adsv2_evt_vr1_e1", "adsv2_evt_vr2a", "adsv2_evt_vr2b", "adsv2_evt_vr3b_f1", "adsv2_evt_vr3a", "adsv2_evt_vr3b", "adsv2_evt_vr3_o1", "adsv2_evt_vr4", "adsv2_evt_vr5", "adsv2_evt_vr3_b1_1", "adsv2_evt_vr3_b1_1_f1", "adsv2_evt_vr3_b1_2", "adsv2_evt_vr3_b1_2_f1", "adsv2_evt_vr3_b1_o1", "adsv2_evt_vr3_b1_o3", "adsv2_evt_vr4_b2", "adsv2_evt_vr4_b2_f1", "adsv2_evt_vr4_b2_4", "adsv2_evt_vr4_b2_1", "adsv2_evt_vr4_b2_2", "adsv2_evt_vr4_b2_5", "adsv2_evt_vr4_b2_3", "adsv2_evt_vr4_b2_6", "adsv2_evt_vr4_b2_o1", "adsv2_evt_vr6_b1_1", "adsv2_evt_va1", "adsv2_evt_va0", "adsv2_evt_va0_f1", "adsv2_evt_va5", "adsv2_evt_va5_b1_4", "adsv2_evt_va6", "adsv2_evt_va6_b1_f1", "adsv2_evt_va6_b1_1", "adsv2_evt_va6_b1_4", "adsv2_evt_va6_b1_2", "adsv2_evt_va6_b1_3", "adsv2_evt_va6_b1_5", "adsv2_evt_va6_b1_6", "adsv2_evt_va6_b1_o1", "adsv2_evt_va7", "adsv2_evt_va7_b1_1", "adsv2_evt_va7_b1_2", "adsv2_evt_va7_b1_3", "adsv2_evt_va7_b1_5", "adsv2_evt_vm1", "adsv2_evt_vm1_f1", "adsv2_evt_vm3", "adsv2_evt_vm4c", "adsv2_evt_vm2_o1", "adsv2_evt_vm6", "adsv2_evt_vm6_b1_1", "adsv2_evt_vm6_b1_2", "adsv2_evt_vm3_b1_1", "adsv2_evt_vm3_b1_2", "adsv2_evt_vm3_b1_3", "adsv2_evt_vm3_b1_6", "adsv2_evt_vm3_b1_4", "adsv2_evt_vm3_b1_5", "adsv2_evt_vm3_b1_f1", "adsv2_evt_vm3_b1_o1", "adsv2_evt_vc1a", "adsv2_evt_vc2", "adsv2_evt_vc3", "adsv2_evt_vc4", "adsv2_evt_vc5", "adsv2_evt_vc6", "adsv2_evt_varr_1", "adsv2_evt_varr_2", "adsv2_evt_varr_3", "adsv2_evt_varr_4", "adsv2_evt_varr_o1", "adsv2_evt_varr_o2", "adsv2_evt_vcl1a", "adsv2_evt_vcl1b", "adsv2_evt_vcl1c", "adsv2_evt_vd1", "adsv2_evt_vd1_f1", "adsv2_evt_vd3_f1", "adsv2_evt_vd_v1"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("17_Van", "succeeded")
